# Google Play Store Ratings Analysis


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 30)

## 1. Load the dataset

In [ ]:
data_path = Path("googleplaystore.csv")

# Fallback used when this notebook runs in the provided environment.
if not data_path.exists():
    data_path = Path("/mnt/data/googleplaystore.csv")

raw = pd.read_csv(data_path)

print(f"Raw rows: {len(raw):,}")
print(f"Columns: {len(raw.columns)}")
raw.head()

## 2. Convert fields into numeric values

Reviews, installs, size, and price are stored as text in the original file. They must be converted before they can be summarized or correlated with rating.

In [ ]:
data = raw.copy()

data["Rating_num"] = pd.to_numeric(data["Rating"], errors="coerce")

data["Reviews_num"] = pd.to_numeric(
    data["Reviews"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False),
    errors="coerce"
)

data["Installs_num"] = pd.to_numeric(
    data["Installs"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False),
    errors="coerce"
)

data["Price_usd"] = pd.to_numeric(
    data["Price"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
    errors="coerce"
)

def size_to_mb(value):
    value = str(value).strip()

    if value in {"Varies with device", "nan"}:
        return np.nan
    if value.endswith("M"):
        return float(value[:-1])
    if value.endswith("k"):
        return float(value[:-1]) / 1024

    return np.nan

data["Size_mb"] = data["Size"].apply(size_to_mb)
data["Last_updated_date"] = pd.to_datetime(
    data["Last Updated"],
    errors="coerce"
)

data[
    ["App", "Rating_num", "Reviews_num", "Installs_num", "Size_mb", "Price_usd"]
].head()

## 3. Remove the malformed row

In [ ]:
# Valid Google Play ratings are between 0 and 5.
malformed_rows = data[
    data["Rating_num"].notna()
    & ~data["Rating_num"].between(0, 5)
]

print(f"Malformed rows found: {len(malformed_rows)}")
malformed_rows[
    ["App", "Category", "Rating", "Reviews", "Size", "Installs"]
]

In [ ]:
clean = data.drop(malformed_rows.index).copy()

print(f"Rows after removing malformed data: {len(clean):,}")

## 4. Remove duplicate app names

When an app appears more than once, keep the row with a valid rating, then the largest review count, then the most recent update date.

In [ ]:
clean["Has_rating"] = clean["Rating_num"].notna()

clean = clean.sort_values(
    ["App", "Has_rating", "Reviews_num", "Last_updated_date"],
    ascending=[True, False, False, False]
)

unique_apps = clean.drop_duplicates(
    subset="App",
    keep="first"
).copy()

rated_apps = unique_apps[
    unique_apps["Rating_num"].notna()
].copy()

print(f"Unique apps: {len(unique_apps):,}")
print(f"Duplicate rows removed: {len(clean) - len(unique_apps):,}")
print(f"Unique apps without ratings: {unique_apps['Rating_num'].isna().sum():,}")
print(f"Unique rated apps used in analysis: {len(rated_apps):,}")

## 5. Descriptive statistics

In [ ]:
rating_summary = rated_apps["Rating_num"].describe().rename(
    {
        "count": "Count",
        "mean": "Mean",
        "std": "Standard deviation",
        "min": "Minimum",
        "25%": "First quartile",
        "50%": "Median",
        "75%": "Third quartile",
        "max": "Maximum",
    }
)

rating_summary.to_frame("Rating")

In [ ]:
print(f"Mean rating: {rated_apps['Rating_num'].mean():.2f}")
print(f"Median rating: {rated_apps['Rating_num'].median():.2f}")
print(f"Standard deviation: {rated_apps['Rating_num'].std():.2f}")
print(
    "Middle 50% of ratings: "
    f"{rated_apps['Rating_num'].quantile(.25):.1f} to "
    f"{rated_apps['Rating_num'].quantile(.75):.1f}"
)

## 6. Pearson correlations

Review and install counts are highly uneven, so the analysis uses `log10(x + 1)` for those two variables. Size and price are used without a log transformation.

In [ ]:
rated_apps["Log_reviews"] = np.log10(
    rated_apps["Reviews_num"] + 1
)

rated_apps["Log_installs"] = np.log10(
    rated_apps["Installs_num"] + 1
)

paid_apps = rated_apps[
    rated_apps["Price_usd"] > 0
].copy()

correlations = pd.Series(
    {
        "Log review count": rated_apps["Rating_num"].corr(
            rated_apps["Log_reviews"]
        ),
        "Log install count": rated_apps["Rating_num"].corr(
            rated_apps["Log_installs"]
        ),
        "App size (MB)": rated_apps["Rating_num"].corr(
            rated_apps["Size_mb"]
        ),
        "Price (all apps)": rated_apps["Rating_num"].corr(
            rated_apps["Price_usd"]
        ),
        "Price (paid apps only)": paid_apps["Rating_num"].corr(
            paid_apps["Price_usd"]
        ),
    },
    name="Pearson r"
)

correlations.to_frame().round(3)

### Figure 1. Numeric correlations with rating

In [ ]:
plt.figure(figsize=(9, 5))
correlations.plot(kind="bar")
plt.axhline(0, linewidth=1)
plt.ylabel("Pearson correlation with rating (r)")
plt.xlabel("Variable")
plt.title("Figure 1. Numeric Variables and Google Play Rating")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 7. Average rating by install range

In [ ]:
install_bins = [
    -np.inf,
    10_000,
    100_000,
    1_000_000,
    10_000_000,
    np.inf,
]

install_labels = [
    "<10,000",
    "10,000-99,999",
    "100,000-999,999",
    "1,000,000-9,999,999",
    ">=10,000,000",
]

rated_apps["Install_range"] = pd.cut(
    rated_apps["Installs_num"],
    bins=install_bins,
    labels=install_labels,
    right=False
)

install_summary = (
    rated_apps
    .groupby("Install_range", observed=False)["Rating_num"]
    .agg(["count", "mean", "std"])
    .rename(
        columns={
            "count": "App count",
            "mean": "Mean rating",
            "std": "Rating SD",
        }
    )
)

install_summary["Standard error"] = (
    install_summary["Rating SD"]
    / np.sqrt(install_summary["App count"])
)

install_summary["95% CI"] = (
    1.96 * install_summary["Standard error"]
)

install_summary.round(3)

### Figure 2. Average rating by install range

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(
    install_summary.index.astype(str),
    install_summary["Mean rating"],
    yerr=install_summary["95% CI"],
    capsize=4
)
plt.ylabel("Average rating")
plt.xlabel("Install range")
plt.title("Figure 2. Average Rating by Install Range")
plt.ylim(3.8, 4.5)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 8. Free and paid app comparison

In [ ]:
price_type_summary = (
    rated_apps
    .assign(
        Price_model=np.where(
            rated_apps["Price_usd"] > 0,
            "Paid",
            "Free"
        )
    )
    .groupby("Price_model")["Rating_num"]
    .agg(["count", "mean", "median", "std"])
    .rename(
        columns={
            "count": "App count",
            "mean": "Mean rating",
            "median": "Median rating",
            "std": "Rating SD",
        }
    )
)

price_type_summary.round(3)

In [ ]:
free_mean = price_type_summary.loc["Free", "Mean rating"]
paid_mean = price_type_summary.loc["Paid", "Mean rating"]

print(f"Free apps mean rating: {free_mean:.2f}")
print(f"Paid apps mean rating: {paid_mean:.2f}")
print(f"Difference: {paid_mean - free_mean:.2f} stars")
print(
    "Price correlation among paid apps: "
    f"{correlations['Price (paid apps only)']:.3f}"
)

## 9. Average rating by app category

In [ ]:
category_summary = (
    rated_apps
    .groupby("Category")["Rating_num"]
    .agg(["count", "mean"])
    .rename(
        columns={
            "count": "App count",
            "mean": "Mean rating",
        }
    )
)

# Keep categories with at least 100 rated apps.
large_categories = category_summary[
    category_summary["App count"] >= 100
].sort_values("Mean rating", ascending=False)

large_categories.round(3)

### Figure 3. Average ratings for categories with at least 100 apps

In [ ]:
category_plot = large_categories.sort_values(
    "Mean rating",
    ascending=True
)

plt.figure(figsize=(9, 8))
plt.barh(
    category_plot.index,
    category_plot["Mean rating"]
)
plt.xlabel("Average rating")
plt.ylabel("Category")
plt.title(
    "Figure 3. Average Rating by Category\n"
    "(Categories with at Least 100 Rated Apps)"
)
plt.xlim(3.9, 4.4)
plt.tight_layout()
plt.show()

## 10. Average rating by content rating

In [ ]:
content_rating_summary = (
    rated_apps
    .groupby("Content Rating")["Rating_num"]
    .agg(["count", "mean", "median"])
    .rename(
        columns={
            "count": "App count",
            "mean": "Mean rating",
            "median": "Median rating",
        }
    )
    .sort_values("Mean rating", ascending=False)
)

content_rating_summary.round(3)

## 11. Main findings

In [ ]:
print("MAIN RESULTS")
print("-" * 50)
print(f"Unique rated apps analyzed: {len(rated_apps):,}")
print(f"Average rating: {rated_apps['Rating_num'].mean():.2f}")
print(
    "Review count correlation: "
    f"{correlations['Log review count']:.3f}"
)
print(
    "Install count correlation: "
    f"{correlations['Log install count']:.3f}"
)
print(
    "App size correlation: "
    f"{correlations['App size (MB)']:.3f}"
)
print(
    "Price correlation among paid apps: "
    f"{correlations['Price (paid apps only)']:.3f}"
)
print(
    f"Free app average: {free_mean:.2f}; "
    f"paid app average: {paid_mean:.2f}"
)
print()
print(
    "Conclusion: Review count had the strongest relationship "
    "with rating, but all measured relationships were weak."
)

## 12. Optional: save the cleaned dataset

This cell saves the 8,196 unique rated apps used in the analysis.

In [ ]:
output_columns = [
    "App",
    "Category",
    "Rating_num",
    "Reviews_num",
    "Installs_num",
    "Size_mb",
    "Type",
    "Price_usd",
    "Content Rating",
]

output_name = "googleplay_cleaned_unique_rated_notebook.csv"
output_folder = Path("/mnt/data") if Path("/mnt/data").exists() else Path.cwd()
cleaned_output_path = output_folder / output_name

rated_apps[output_columns].to_csv(
    cleaned_output_path,
    index=False
)

print(f"Saved: {cleaned_output_path}")